# Threshold Tuning: Does Weighted Argmax Beat Plain Argmax?

Our best model so far is a CatBoost classifier trained on an engineered feature
set (missingness indicators, categorical crosses, and out-of-fold smoothed
multiclass target encoding): OOF balanced accuracy **0.9491**, public LB
**0.94913**. A version trained on the base features alone scores OOF **0.9493**
/ LB **0.94885** — the two are statistically tied, and this notebook builds on
the engineered-feature version since it scored slightly higher on the public
leaderboard.

Balanced accuracy is the mean of per-class recall. Argmax over raw softmax
probabilities is not guaranteed to maximize that specific objective under class
imbalance, even with `auto_class_weights='Balanced'` during training. This
notebook checks whether a **weighted argmax** — `predict = argmax(proba * w)`
with `w` tuned per class — beats plain argmax.

There may be limited headroom here to begin with: CatBoost's per-class recalls
are already fairly balanced (at-risk 0.934, fit 0.949, unhealthy 0.964). Still,
this is a cheap, principled check to run either way — a negative result is as
useful to know as a positive one.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

import glob

DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not (DATA_DIR / 'train.csv').exists():
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(hits[0]).parent
    else:
        DATA_DIR = Path('..') / 'data'  # local fallback for testing outside Kaggle

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print('train', train.shape, 'test', test.shape)

classes: ['at-risk', 'fit', 'unhealthy']
train (690088, 15) test (295753, 14)


## Reconstruct v0.2/v0.3 Variant 2's engineered feature set

Same as v0.3 Variant 2: missingness indicators, categorical crosses, OOF smoothed
multiclass target encoding.

In [2]:
train['sleep_duration_bin'], sleep_bin_edges = pd.qcut(
    train['sleep_duration'], q=10, duplicates='drop', retbins=True)
test_sleep_clipped = test['sleep_duration'].clip(lower=sleep_bin_edges[0], upper=sleep_bin_edges[-1])
test['sleep_duration_bin'] = pd.cut(test_sleep_clipped, bins=sleep_bin_edges, include_lowest=True)

train['sleep_duration_bin'] = train['sleep_duration_bin'].astype(str)
train.loc[train['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'
test['sleep_duration_bin'] = test['sleep_duration_bin'].astype(str)
test.loc[test['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'

MISSING_INDICATOR_COLS = []
for col in NUMERIC_COLS:
    ind_col = f'is_missing_{col}'
    train[ind_col] = train[col].isnull().astype(int)
    test[ind_col] = test[col].isnull().astype(int)
    MISSING_INDICATOR_COLS.append(ind_col)

def make_cross(df, col_a, col_b, name):
    df[name] = df[col_a].astype(str) + '_' + df[col_b].astype(str)

make_cross(train, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(test, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(train, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(test, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(train, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')
make_cross(test, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')

CROSS_COLS = ['stress_x_activity', 'sleepq_x_smoking', 'sleepbin_x_stress']

def oof_target_encode(train_col, test_col, y, n_classes, folds, alpha=20):
    n_train = len(train_col)
    oof_encoded = np.zeros((n_train, n_classes))
    test_encoded = np.zeros((len(test_col), n_classes))
    class_cols = [f'k{i}' for i in range(n_classes)]
    onehot = pd.get_dummies(pd.Series(y)).values
    for tr_idx, val_idx in folds:
        cats_tr = pd.Series(train_col.iloc[tr_idx].astype(str).values)
        y_tr_onehot = onehot[tr_idx]
        prior = pd.Series(y_tr_onehot.mean(axis=0), index=class_cols)
        stats = pd.DataFrame(y_tr_onehot, columns=class_cols)
        stats['cat'] = cats_tr.values
        grp_sum = stats.groupby('cat')[class_cols].sum()
        grp_count = stats.groupby('cat').size()
        enc_map = grp_sum.add(alpha * prior, axis=1).div(grp_count + alpha, axis=0)
        val_cats = pd.Series(train_col.iloc[val_idx].astype(str).values)
        oof_encoded[val_idx] = enc_map.reindex(val_cats).fillna(prior).values
        test_cats = pd.Series(test_col.astype(str).values)
        test_encoded += enc_map.reindex(test_cats).fillna(prior).values / len(folds)
    return oof_encoded, test_encoded

TARGET_ENCODE_COLS = ['stress_level', 'physical_activity_level', 'stress_x_activity', 'sleepq_x_smoking']
TARGET_ENCODED_FEATURE_COLS = []
for col in tqdm(TARGET_ENCODE_COLS, desc='target encoding'):
    oof_enc, test_enc = oof_target_encode(train[col], test[col], y, n_classes, folds)
    for k in range(n_classes):
        fcol = f'te_{col}_k{k}'
        train[fcol] = oof_enc[:, k]
        test[fcol] = test_enc[:, k]
        TARGET_ENCODED_FEATURE_COLS.append(fcol)

ENGINEERED_CATEGORICAL_COLS = CATEGORICAL_COLS + CROSS_COLS
ENGINEERED_FEATURES = (NUMERIC_COLS + CATEGORICAL_COLS + MISSING_INDICATOR_COLS
                        + CROSS_COLS + TARGET_ENCODED_FEATURE_COLS)
print(f'{len(ENGINEERED_FEATURES)} engineered features (matches v0.2/v0.3 Variant 2)')

target encoding:   0%|          | 0/4 [00:00<?, ?it/s]

35 engineered features (matches v0.2/v0.3 Variant 2)


## Reproduce v0.3 Variant 2, this time capturing OOF probabilities

v0.3's harness only stored hard `predict()` labels per fold, not the probability
matrix needed for threshold tuning. Re-run the identical config (same features,
folds, hyperparameters, `random_seed=42`) and capture `predict_proba` on each
validation fold instead. CatBoost on CPU with a fixed seed should reproduce closely;
sanity-checked against v0.3's known results below before trusting anything downstream.

In [3]:
class BalancedAccuracyMetric:
    def is_max_optimal(self):
        return True
    def evaluate(self, approxes, target, weight):
        approx = np.array(approxes)
        pred = approx.argmax(axis=0)
        y_true = np.array(target).astype(int)
        return balanced_accuracy_score(y_true, pred), 1.0
    def get_final_error(self, error, weight):
        return error

class TqdmCatBoostCallback:
    def __init__(self, total, desc):
        self.pbar = tqdm(total=total, desc=desc, leave=False)
        self.last = 0
    def after_iteration(self, info):
        self.pbar.update(info.iteration - self.last)
        self.last = info.iteration
        return True

def run_catboost_cv_with_proba(feature_frame, test_frame, categorical_features, folds, y, n_classes,
                                desc='folds', iterations=3000, learning_rate=0.05, depth=6,
                                l2_leaf_reg=3, early_stopping_rounds=100):
    oof_proba = np.zeros((len(feature_frame), n_classes))
    test_proba = np.zeros((len(test_frame), n_classes))
    fold_scores, best_iters, models = [], [], []
    for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc=desc)):
        X_tr, X_val = feature_frame.iloc[tr_idx], feature_frame.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric=BalancedAccuracyMetric(),
            auto_class_weights='Balanced',
            iterations=iterations,
            learning_rate=learning_rate,
            depth=depth,
            l2_leaf_reg=l2_leaf_reg,
            random_seed=42,
            task_type='CPU',
            verbose=False,
        )
        round_pbar = TqdmCatBoostCallback(iterations, f'{desc} fold {fold} rounds')
        model.fit(X_tr, y_tr, cat_features=categorical_features, eval_set=(X_val, y_val),
                  early_stopping_rounds=early_stopping_rounds, callbacks=[round_pbar])
        round_pbar.pbar.close()
        oof_proba[val_idx] = model.predict_proba(X_val)
        val_pred = oof_proba[val_idx].argmax(axis=1)
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))
        best_iters.append(model.best_iteration_)
        test_proba += model.predict_proba(test_frame) / len(folds)
        models.append(model)
    oof_pred = oof_proba.argmax(axis=1)
    oof_bal_acc = balanced_accuracy_score(y, oof_pred)
    return {'oof_proba': oof_proba, 'oof_pred': oof_pred, 'test_proba': test_proba,
            'fold_scores': fold_scores, 'best_iters': best_iters,
            'oof_balanced_accuracy': oof_bal_acc, 'models': models}

result = run_catboost_cv_with_proba(train[ENGINEERED_FEATURES], test[ENGINEERED_FEATURES],
                                     ENGINEERED_CATEGORICAL_COLS, folds, y, n_classes, desc='v0.4 repro')

print('best_iterations:', result['best_iters'])
print('fold scores:', [round(s, 4) for s in result['fold_scores']])
print(f"OOF balanced accuracy (plain argmax): {result['oof_balanced_accuracy']:.4f}")
print("v0.3 Variant 2 known result:          0.9491, best_iterations [544, 765, 542, 354, 628]")

v0.4 repro:   0%|          | 0/5 [00:00<?, ?it/s]

v0.4 repro fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_37188/3468950682.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


v0.4 repro fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_37188/3468950682.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


...

v0.4 repro fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_37188/3468950682.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


v0.4 repro fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_37188/3468950682.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


best_iterations: [544, 765, 542, 354, 628]
fold scores: [0.9498, 0.9511, 0.9482, 0.9486, 0.9481]
OOF balanced accuracy (plain argmax): 0.9491
v0.3 Variant 2 known result:          0.9491, best_iterations [544, 765, 542, 354, 628]


In [4]:
reproduced_closely = abs(result['oof_balanced_accuracy'] - 0.9491) < 0.002
print(f"Reproduction check: {'PASS' if reproduced_closely else 'FAIL — investigate before trusting downstream results'}")
assert reproduced_closely, 'OOF did not reproduce v0.3 Variant 2 closely enough to trust the probability matrix'
print()
print(classification_report(y, result['oof_pred'], target_names=label_encoder.classes_, digits=4))

Reproduction check: PASS

              precision    recall  f1-score   support

     at-risk     0.9935    0.9341    0.9629    592561
         fit     0.7256    0.9490    0.8224     39803
   unhealthy     0.6880    0.9644    0.8031     57724

    accuracy                         0.9375    690088
   macro avg     0.8024    0.9491    0.8628    690088
weighted avg     0.9525    0.9375    0.9414    690088



## Weighted-argmax grid search

`predict = argmax(proba * w)`, `w[at-risk]` fixed at 1.0 (only ratios matter for
argmax), `w[fit]` and `w[unhealthy]` searched on a log scale to directly maximize
OOF balanced accuracy.

In [5]:
CLASSES = list(label_encoder.classes_)
AT_RISK_IDX = CLASSES.index('at-risk')
FIT_IDX = CLASSES.index('fit')
UNHEALTHY_IDX = CLASSES.index('unhealthy')

def weighted_argmax(proba, w_fit, w_unhealthy):
    w = np.ones(3)
    w[FIT_IDX] = w_fit
    w[UNHEALTHY_IDX] = w_unhealthy
    return (proba * w).argmax(axis=1)

def grid_search_weights(proba, y_true, grid):
    best_score, best_w = -1, (1.0, 1.0)
    results = []
    for w_fit in tqdm(grid, desc='grid search (fit weight)'):
        for w_unhealthy in grid:
            pred = weighted_argmax(proba, w_fit, w_unhealthy)
            score = balanced_accuracy_score(y_true, pred)
            results.append((w_fit, w_unhealthy, score))
            if score > best_score:
                best_score, best_w = score, (w_fit, w_unhealthy)
    return best_score, best_w, results

grid = np.logspace(-1, 1, 41)  # 0.1 to 10, 41 points
best_score_full, best_w_full, grid_results = grid_search_weights(result['oof_proba'], y, grid)
print(f'Full-OOF grid search: best balanced accuracy {best_score_full:.4f} at w_fit={best_w_full[0]:.3f}, w_unhealthy={best_w_full[1]:.3f}')
print(f'Plain argmax (w=1,1):                 {result["oof_balanced_accuracy"]:.4f}')
print(f'Improvement (same-data, likely optimistic): {best_score_full - result["oof_balanced_accuracy"]:+.4f}')

grid search (fit weight):   0%|          | 0/41 [00:00<?, ?it/s]

Full-OOF grid search: best balanced accuracy 0.9491 at w_fit=1.000, w_unhealthy=1.000
Plain argmax (w=1,1):                 0.9491
Improvement (same-data, likely optimistic): +0.0000


## Nested validation

Fit weights on 4/5 folds, evaluate on the held-out 5th, cycle across all 5 folds.
This is the honest estimate — the full-OOF grid search above optimizes 2 parameters
directly on the same data used to report the score, which is mildly optimistic even
with only 2 free parameters and 690k rows.

In [6]:
nested_scores_plain = []
nested_scores_tuned = []
nested_weights = []

for fold_idx, (_, val_idx) in enumerate(tqdm(folds, desc='nested CV')):
    other_idx = np.concatenate([folds[j][1] for j in range(N_FOLDS) if j != fold_idx])
    _, w_fold, _ = grid_search_weights(result['oof_proba'][other_idx], y[other_idx], grid)

    plain_pred = result['oof_proba'][val_idx].argmax(axis=1)
    tuned_pred = weighted_argmax(result['oof_proba'][val_idx], *w_fold)

    nested_scores_plain.append(balanced_accuracy_score(y[val_idx], plain_pred))
    nested_scores_tuned.append(balanced_accuracy_score(y[val_idx], tuned_pred))
    nested_weights.append(w_fold)

print('Per-fold weights found (on the other 4 folds):', [tuple(round(x, 3) for x in w) for w in nested_weights])
print(f'Nested plain-argmax balanced accuracy: {np.mean(nested_scores_plain):.4f} (+/- {np.std(nested_scores_plain):.4f})')
print(f'Nested tuned-argmax balanced accuracy: {np.mean(nested_scores_tuned):.4f} (+/- {np.std(nested_scores_tuned):.4f})')
print(f'Honest improvement estimate: {np.mean(nested_scores_tuned) - np.mean(nested_scores_plain):+.4f}')

nested CV:   0%|          | 0/5 [00:00<?, ?it/s]

grid search (fit weight):   0%|          | 0/41 [00:00<?, ?it/s]

grid search (fit weight):   0%|          | 0/41 [00:00<?, ?it/s]

grid search (fit weight):   0%|          | 0/41 [00:00<?, ?it/s]

grid search (fit weight):   0%|          | 0/41 [00:00<?, ?it/s]

grid search (fit weight):   0%|          | 0/41 [00:00<?, ?it/s]

Per-fold weights found (on the other 4 folds): [(np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.413)), (np.float64(1.0), np.float64(1.0)), (np.float64(0.891), np.float64(1.0)), (np.float64(1.0), np.float64(1.0))]
Nested plain-argmax balanced accuracy: 0.9491 (+/- 0.0011)
Nested tuned-argmax balanced accuracy: 0.9490 (+/- 0.0011)
Honest improvement estimate: -0.0001


## Decision + candidate submission

Use the nested-validation estimate (not the full-OOF grid search) to decide whether
threshold tuning is a real improvement. Only write a new `submission.csv` if it is.

In [7]:
honest_improvement = np.mean(nested_scores_tuned) - np.mean(nested_scores_plain)
THRESHOLD_FOR_REAL_IMPROVEMENT = 0.0005  # smaller than this is noise, not signal

if honest_improvement > THRESHOLD_FOR_REAL_IMPROVEMENT:
    print(f'REAL IMPROVEMENT: {honest_improvement:+.4f} (nested-validated). Using tuned weights fit on full OOF: {best_w_full}')
    test_pred = weighted_argmax(result['test_proba'], *best_w_full)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows) — NOT auto-submitted to Kaggle')
    display(submission[TARGET].value_counts(normalize=True))
else:
    print(f'NO REAL IMPROVEMENT: nested-validated delta {honest_improvement:+.4f} is within noise '
          f'(threshold {THRESHOLD_FOR_REAL_IMPROVEMENT}). Keeping v0.3 Variant 1/2 as the best model. '
          f'Not writing a new submission.csv.')

NO REAL IMPROVEMENT: nested-validated delta -0.0001 is within noise (threshold 0.0005). Keeping v0.3 Variant 1/2 as the best model. Not writing a new submission.csv.


## Summary

**Negative result, cleanly explained: threshold tuning does not beat plain argmax.**

- **Reproduction**: exact match against v0.3 Variant 2 (`best_iterations [544, 765,
  542, 354, 628]`, OOF 0.9491) — the OOF probability matrix is trustworthy.
- **Full-OOF grid search**: best weights found were **w=(1.0, 1.0)** — plain argmax
  was already optimal. Zero improvement, even on the same-data fit that's normally
  mildly optimistic in favor of finding *something*.
- **Nested validation** (fit weights on 4/5 folds, evaluate on the held-out 5th,
  cycle across folds): nested plain-argmax 0.9491 (+/- 0.0011), nested tuned-argmax
  0.9490 (+/- 0.0011). **Honest improvement estimate: -0.0001** — within noise, not
  a real effect. No new `submission.csv` written, correctly.
- **Why**: per Kaggle discussion 717018 (Georgy Mamarin), stacking training-time
  class-weighting with a *separate* post-hoc threshold/prior correction is a known
  pitfall that can actively hurt (he measured a drop to 0.9047 from ~0.950 for either
  correction alone, on his own independent pipeline) — the second correction
  double-corrects probabilities the first one already shifted. Our CatBoost models
  were trained with `auto_class_weights='Balanced'`, so the balance correction was
  already applied during training; there was no residual imbalance left for a
  post-hoc weighted argmax to fix. This isn't just an empirical shrug — it's the
  expected outcome given how the model was already trained.
- **Broader context** (same discussion + 717222, broccoli beef): the competition
  data is likely a noised synthesis of a near-deterministic depth-4 decision rule on
  `sleep_duration`/`stress_level`/`physical_activity_level` (100% accuracy on the
  original pre-synthesis dataset, per a `DecisionTreeClassifier` fit directly to it).
  Multiple independent competitor pipelines, across different model families,
  converge to the same ~0.948-0.950 OOF / ~0.9498 LB range — consistent with a
  synthesis-noise ceiling rather than an easy gain being left on the table.

**Decision**: v0.3 (either variant, still statistically tied) remains the best model.
Ensembling is a natural next lever, but expectations should be tempered
given the likely ~0.95 ceiling on this dataset.